In [1]:
"""
descargar UN MES COMPLETO del dataset SUNT OD
desde Hugging Face
"""

from huggingface_hub import hf_hub_download
import pandas as pd
from datetime import datetime, timedelta
import calendar
from tqdm import tqdm  # Para barra de progreso
import warnings
warnings.filterwarnings('ignore')

In [2]:
# ==============================================
# CONFIGURACIÓN - CAMBIAR AQUÍ
# ==============================================

# Mes y año a descargar
AÑO = 2024
MES = 3  # 3 = Marzo, 4 = Abril, etc.

# Configuración de carga
MOSTRAR_PROGRESO = True  # Barra de progreso
VERBOSE = True           # Mensajes detallados
VALIDAR_DATOS = True     # Validación al final

# ==============================================
# FUNCIÓN PRINCIPAL
# ==============================================

def descargar_mes_completo(año, mes):
    """
    Descarga todos los días disponibles de un mes específico

    Parámetros:
    -----------
    año : int
        Año (ej: 2024)
    mes : int
        Mes (1-12)

    Retorna:
    --------
    DataFrame con todos los datos del mes concatenados
    """

    print("=" * 70)
    print(f"🚌 DESCARGANDO DATASET SUNT OD - {calendar.month_name[mes]} {año}")
    print("=" * 70)

    # Calcular número de días en el mes
    num_dias = calendar.monthrange(año, mes)[1]
    print(f"\n📅 Días en el mes: {num_dias}")
    print(f"🎯 Repositorio: labiaufba/PublicTransportationSunt")
    print(f"📂 Carpeta: OD/")

    # Lista para almacenar DataFrames
    dfs = []
    dias_exitosos = []
    dias_fallidos = []

    # Configurar iterador con o sin barra de progreso
    if MOSTRAR_PROGRESO:
        try:
            from tqdm import tqdm
            dias_range = tqdm(range(1, num_dias + 1), desc="Descargando", unit="día")
        except ImportError:
            print("\n💡 Tip: Instala tqdm para barra de progreso: pip install tqdm\n")
            dias_range = range(1, num_dias + 1)
    else:
        dias_range = range(1, num_dias + 1)

    print("\n" + "-" * 70)
    print("📥 INICIANDO DESCARGA")
    print("-" * 70 + "\n")

    # Descargar cada día
    for dia in dias_range:
        fecha = datetime(año, mes, dia)
        fecha_str = fecha.strftime("%Y-%m-%d")
        filename = f"OD/od-{fecha_str}.parquet"

        try:
            # Descargar archivo
            if VERBOSE and not MOSTRAR_PROGRESO:
                print(f"📥 {fecha_str}...", end=" ", flush=True)

            file_path = hf_hub_download(
                repo_id="labiaufba/PublicTransportationSunt",
                filename=filename,
                repo_type="dataset"
            )

            # Cargar con pandas
            df_dia = pd.read_parquet(file_path)

            # Verificar que no esté vacío
            if len(df_dia) == 0:
                if VERBOSE and not MOSTRAR_PROGRESO:
                    print("⚠️  VACÍO")
                dias_fallidos.append(fecha_str)
                continue

            dfs.append(df_dia)
            dias_exitosos.append(fecha_str)

            if VERBOSE and not MOSTRAR_PROGRESO:
                print(f"✅ {len(df_dia):,} registros")

        except Exception as e:
            if VERBOSE and not MOSTRAR_PROGRESO:
                print(f"❌ No disponible")
            dias_fallidos.append(fecha_str)
            continue

    print("\n" + "-" * 70)
    print("📊 RESUMEN DE DESCARGA")
    print("-" * 70)
    print(f"✅ Días descargados exitosamente: {len(dias_exitosos)}/{num_dias}")
    print(f"❌ Días no disponibles/fallidos: {len(dias_fallidos)}/{num_dias}")

    if dias_fallidos and len(dias_fallidos) <= 10:
        print(f"\n⚠️  Días faltantes: {', '.join(dias_fallidos)}")

    # Verificar si se descargó algo
    if not dfs:
        print("\n❌ ERROR: No se pudo descargar ningún día")
        print("💡 Posibles causas:")
        print("   - El mes aún no está disponible en el dataset")
        print("   - Problemas de conexión a internet")
        print("   - El dataset cambió de estructura")
        return None

    # Concatenar todos los DataFrames
    print("\n" + "-" * 70)
    print("🔗 CONCATENANDO DATOS")
    print("-" * 70)

    df = pd.concat(dfs, ignore_index=True)

    print(f"✅ Concatenación exitosa")
    print(f"   Total registros: {len(df):,}")
    print(f"   Total columnas: {len(df.columns)}")
    print(f"   Memoria usada: {df.memory_usage(deep=True).sum() / 1024**2:.2f} MB")

    return df


In [ ]:
# ==============================================
# VALIDACIÓN DE DATOS
# ==============================================

def validar_datos_od(df):
    """
    Valida que los datos sean correctos y completos
    """
    print("\n" + "=" * 70)
    print("🔍 VALIDACIÓN DE DATOS")
    print("=" * 70)

    # Columnas críticas para predicción de ocupación
    columnas_criticas = {
        'loading': 'Ocupación del bus (CRÍTICO)',
        'n_boardings': 'Número de abordajes',
        'n_alighting': 'Número de descensos',
        'route_short_name': 'Línea de bus',
        'stop_id': 'ID de parada',
        'direction_id': 'Dirección del bus',
        'pt_sequence': 'Secuencia de parada',
        'gps_datetime': 'Timestamp',
    }

    print("\n📋 VERIFICANDO COLUMNAS CRÍTICAS:")
    print("-" * 70)

    presentes = []
    faltantes = []

    for col, descripcion in columnas_criticas.items():
        if col in df.columns:
            presentes.append(col)
            print(f"   ✅ {col:20s} - {descripcion}")
        else:
            faltantes.append(col)
            print(f"   ❌ {col:20s} - {descripcion} [FALTANTE]")

    if faltantes:
        print(f"\n⚠️  ADVERTENCIA: Faltan {len(faltantes)} columnas críticas")
        return False

    # Validar columna LOADING (la más importante)
    print("\n" + "-" * 70)
    print("📊 ANÁLISIS DE LOADING (Ocupación)")
    print("-" * 70)

    if 'loading' in df.columns:
        loading_stats = df['loading'].describe()
        print(f"   Registros totales: {len(df):,}")
        print(f"   Valores válidos:   {df['loading'].notna().sum():,}")
        print(f"   Valores NaN:       {df['loading'].isna().sum():,} ({df['loading'].isna().mean()*100:.2f}%)")
        print(f"\n   Estadísticas:")
        print(f"      Media:    {loading_stats['mean']:.2f} pasajeros")
        print(f"      Mediana:  {loading_stats['50%']:.2f} pasajeros")
        print(f"      Mínimo:   {loading_stats['min']:.0f} pasajeros")
        print(f"      Máximo:   {loading_stats['max']:.0f} pasajeros")
        print(f"      Std Dev:  {loading_stats['std']:.2f} pasajeros")

        # Verificar valores negativos
        negativos = (df['loading'] < 0).sum()
        if negativos > 0:
            print(f"\n   ⚠️  {negativos} valores negativos detectados (serán filtrados)")

    # Validar fechas
    print("\n" + "-" * 70)
    print("📅 ANÁLISIS TEMPORAL")
    print("-" * 70)

    if 'gps_datetime' in df.columns:
        # Convertir a datetime si no lo es
        if not pd.api.types.is_datetime64_any_dtype(df['gps_datetime']):
            df['gps_datetime'] = pd.to_datetime(df['gps_datetime'], errors='coerce')

        fecha_min = df['gps_datetime'].min()
        fecha_max = df['gps_datetime'].max()
        dias_span = (fecha_max - fecha_min).days + 1

        print(f"   Fecha inicio: {fecha_min}")
        print(f"   Fecha fin:    {fecha_max}")
        print(f"   Días span:    {dias_span} días")

    # Validar líneas de bus
    print("\n" + "-" * 70)
    print("🚌 ANÁLISIS DE LÍNEAS DE BUS")
    print("-" * 70)

    if 'route_short_name' in df.columns:
        n_lineas = df['route_short_name'].nunique()
        print(f"   Total líneas únicas: {n_lineas}")
        print(f"\n   Top 10 líneas más frecuentes:")
        top_lineas = df['route_short_name'].value_counts().head(10)
        for i, (linea, count) in enumerate(top_lineas.items(), 1):
            pct = (count / len(df)) * 100
            print(f"      {i:2d}. {str(linea):15s} → {count:8,} registros ({pct:5.2f}%)")

    # Validar paradas
    print("\n" + "-" * 70)
    print("🚏 ANÁLISIS DE PARADAS")
    print("-" * 70)

    if 'stop_id' in df.columns:
        n_paradas = df['stop_id'].nunique()
        print(f"   Total paradas únicas: {n_paradas}")

        # Paradas más frecuentes
        top_paradas = df['stop_id'].value_counts().head(5)
        print(f"\n   Top 5 paradas más frecuentes:")
        for i, (parada, count) in enumerate(top_paradas.items(), 1):
            print(f"      {i}. Parada {parada}: {count:,} registros")

    print("\n" + "=" * 70)
    print("✅ VALIDACIÓN COMPLETADA")
    print("=" * 70)

    return len(faltantes) == 0



In [ ]:
# ==============================================
# PREPARACIÓN DE DATOS
# ==============================================

def preparar_datos(df):
    """
    Prepara el DataFrame para análisis/modelado
    """
    print("\n" + "=" * 70)
    print("🔧 PREPARANDO DATOS PARA MODELADO")
    print("=" * 70)

    df = df.copy()
    n_original = len(df)

    # 1. Convertir fechas
    print("\n1️⃣  Convirtiendo columnas de fecha a datetime...")
    date_columns = ['gps_datetime', 'stop_time', 'start_trip']
    for col in date_columns:
        if col in df.columns:
            if not pd.api.types.is_datetime64_any_dtype(df[col]):
                df[col] = pd.to_datetime(df[col], errors='coerce')
                print(f"   ✅ {col} convertido")

    # 2. Ordenar por tiempo
    print("\n2️⃣  Ordenando por timestamp...")
    if 'gps_datetime' in df.columns:
        df = df.sort_values(['route_short_name', 'direction_id', 'start_trip', 'pt_sequence'])
        df = df.reset_index(drop=True)
        print(f"   ✅ Ordenado por ruta, dirección, viaje y secuencia")

    # 3. Filtrar valores inválidos
    print("\n3️⃣  Filtrando valores inválidos...")

    # Eliminar loading negativos
    if 'loading' in df.columns:
        n_antes = len(df)
        df = df[df['loading'] >= 0]
        n_despues = len(df)
        if n_antes != n_despues:
            print(f"   ⚠️  Eliminados {n_antes - n_despues:,} registros con loading negativo")

    # Eliminar filas con NaN en columnas críticas
    columnas_criticas = ['loading', 'route_short_name', 'stop_id']
    columnas_existentes = [c for c in columnas_criticas if c in df.columns]
    n_antes = len(df)
    df = df.dropna(subset=columnas_existentes)
    n_despues = len(df)
    if n_antes != n_despues:
        print(f"   ⚠️  Eliminados {n_antes - n_despues:,} registros con NaN en columnas críticas")

    # 4. Resumen final
    print("\n" + "-" * 70)
    print("📊 RESUMEN FINAL")
    print("-" * 70)
    print(f"   Registros originales: {n_original:,}")
    print(f"   Registros finales:    {len(df):,}")
    print(f"   Registros removidos:  {n_original - len(df):,} ({(n_original - len(df))/n_original*100:.2f}%)")
    print(f"   Shape final:          {df.shape}")
    print(f"   Memoria:              {df.memory_usage(deep=True).sum() / 1024**2:.2f} MB")

    return df



In [ ]:
# ==============================================
# GUARDAR DATOS (OPCIONAL)
# ==============================================

def guardar_datos(df, año, mes, formato='parquet'):
    """
    Guarda el DataFrame procesado en disco

    Parámetros:
    -----------
    df : DataFrame
    año : int
    mes : int
    formato : str
        'parquet' (recomendado), 'csv', o 'pickle'
    """
    nombre_mes = calendar.month_name[mes].lower()
    filename = f"sunt_od_{año}_{mes:02d}_{nombre_mes}.{formato}"

    print(f"\n💾 Guardando datos en {filename}...")

    if formato == 'parquet':
        df.to_parquet(filename, index=False)
    elif formato == 'csv':
        df.to_csv(filename, index=False)
    elif formato == 'pickle':
        df.to_pickle(filename)
    else:
        print(f"❌ Formato desconocido: {formato}")
        return None

    import os
    tamaño_mb = os.path.getsize(filename) / 1024**2
    print(f"✅ Guardado exitosamente ({tamaño_mb:.2f} MB)")
    print(f"📁 Ubicación: {os.path.abspath(filename)}")

    return filename


In [ ]:
# ==============================================
# FUNCIÓN PRINCIPAL COMPLETA
# ==============================================

def main(año=AÑO, mes=MES, guardar=False, formato='parquet'):
    """
    Función principal que ejecuta todo el proceso

    Parámetros:
    -----------
    año : int
        Año a descargar
    mes : int
        Mes a descargar (1-12)
    guardar : bool
        Si True, guarda el DataFrame en disco
    formato : str
        Formato para guardar: 'parquet', 'csv', 'pickle'

    Retorna:
    --------
    DataFrame con todos los datos del mes
    """

    # 1. Descargar datos
    df = descargar_mes_completo(año, mes)

    if df is None:
        print("\n❌ ERROR: No se pudieron descargar datos")
        return None

    # 2. Validar datos
    if VALIDAR_DATOS:
        es_valido = validar_datos_od(df)
        if not es_valido:
            print("\n⚠️  ADVERTENCIA: Algunos datos no pasaron la validación")
            print("    Revisa los mensajes arriba")

    # 3. Preparar datos
    df = preparar_datos(df)

    # 4. Guardar si se solicita
    if guardar:
        guardar_datos(df, año, mes, formato)

    # 5. Mensaje final
    print("\n" + "=" * 70)
    print("🎉 PROCESO COMPLETADO EXITOSAMENTE")
    print("=" * 70)
    print("\n💡 PRÓXIMOS PASOS:")
    print("   1. Aplicar feature engineering:")
    print("      from sunt_feature_engineering import SUNTFeatureEngineering")
    print("      fe = SUNTFeatureEngineering(vehicle_capacity=80)")
    print("      df_processed = fe.fit_transform(df, occupancy_method='categorical')")
    print("\n   2. Entrenar modelo:")
    print("      X, y, features, _ = fe.prepare_for_modeling(df_processed)")
    print("      from sklearn.ensemble import RandomForestClassifier")
    print("      model = RandomForestClassifier()")
    print("      model.fit(X_train, y_train)")
    print("\n   3. Evaluar y optimizar")
    print("=" * 70)

    return df

In [ ]:
# ==============================================
# EJECUCIÓN
# ==============================================

if __name__ == "__main__":

    # Verificar dependencias
    try:
        import pandas as pd
        from huggingface_hub import hf_hub_download
    except ImportError as e:
        print("❌ ERROR: Falta instalar dependencias")
        print("\nEjecuta:")
        print("pip install pandas pyarrow huggingface_hub tqdm")
        exit(1)

    # Ejecutar descarga
    df = main(
        año=AÑO,
        mes=MES,
        guardar=True,  # Cambiar a True para guardar en disco
        formato='parquet'  # 'parquet', 'csv', o 'pickle'
    )

    # El DataFrame está listo en la variable 'df'
    if df is not None:
        print(f"\n✅ DataFrame disponible en variable 'df'")
        print(f"   Usa: df.head() para ver los primeros registros")
        print(f"   Usa: df.info() para ver información del DataFrame")

🚌 DESCARGANDO DATASET SUNT OD - March 2024

📅 Días en el mes: 31
🎯 Repositorio: labiaufba/PublicTransportationSunt
📂 Carpeta: OD/


Descargando:   0%|          | 0/31 [00:00<?, ?día/s]


----------------------------------------------------------------------
📥 INICIANDO DESCARGA
----------------------------------------------------------------------



Descargando: 100%|██████████| 31/31 [00:25<00:00,  1.20día/s]



----------------------------------------------------------------------
📊 RESUMEN DE DESCARGA
----------------------------------------------------------------------
✅ Días descargados exitosamente: 31/31
❌ Días no disponibles/fallidos: 0/31

----------------------------------------------------------------------
🔗 CONCATENANDO DATOS
----------------------------------------------------------------------
✅ Concatenación exitosa
   Total registros: 19,517,947
   Total columnas: 16
   Memoria usada: 7584.56 MB

🔍 VALIDACIÓN DE DATOS

📋 VERIFICANDO COLUMNAS CRÍTICAS:
----------------------------------------------------------------------
   ✅ loading              - Ocupación del bus (CRÍTICO)
   ❌ n_boardings          - Número de abordajes [FALTANTE]
   ❌ n_alighting          - Número de descensos [FALTANTE]
   ✅ route_short_name     - Línea de bus
   ✅ stop_id              - ID de parada
   ✅ direction_id         - Dirección del bus
   ✅ pt_sequence          - Secuencia de parada
   ❌ gps_da